# 多核切分与动态 shape

这一节把 `C=A@B` 扩展到 `M=500,N=300,K=128`。每个核仍使用 128×128 的 Tile，但整幅输出需要多个 Tile 协作完成，最后一行和一列还不满 128。我们依次回答：需要多少任务、每个核领哪些任务、尾块只读写哪些元素，以及这些参数何时确定。


## 环境准备

运行代码前，请完成 [01.06 环境配置](../01_pypto_pro_introduction/01.06_env_and_first_kernel.ipynb)。

在本节所在目录启动 Notebook，按顺序执行单元即可；不需要先运行其他 Notebook。设备号默认取 `TILE_FWK_DEVICE_ID`，未设置时使用 0。


In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)


## 1. Tiling 的两个层次

<img style="display:block; margin-left:0; margin-right:auto;" src="./images/pro_tiling_two_levels.svg" alt="Tiling 的两个层次" width="700px">

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">层次</th><th style="text-align:left;">含义</th><th style="text-align:left;">本课程位置</th></tr>
<tr><td><b>核间切分</b></td><td>数据划分为基本任务块，一组任务块分给每个核</td><td>本节</td></tr>
<tr><td><b>核内切分</b></td><td>选择 Tile 尺寸、K 分块、数据复用和 Buffer 槽位轮转</td><td>第 2 章 02.05</td></tr>
</table>

设计时的五个考虑：**负载均衡**（控制各核工作量与空闲核）、**片上容量**（Tile 放得下并为双缓冲留空间）、**接口约束**（对齐/layout/分形）、**数据复用**（复用数据尽量同核）、**尾块处理**（向上取整 + 有效形状）。


把这些考虑代入本例：M/N 方向的每个输出 Tile 是一个独立任务；K 恰好等于 Tile_K，只需一次矩阵乘。每个任务独占其输出区域，不需要跨核归约或原子累加。后续若沿 K 分给多个核，同一个输出就会产生多个部分和，必须另行设计归约，本例不使用这种切法。


## 2. 把 500×300 的输出变成任务编号

每个任务最多写 128×128 个元素：

- `m_tiles = ceil(500/128) = 4`。
- `n_tiles = ceil(300/128) = 3`。
- `total = 4×3 = 12`。

行优先编号 `task = i*n_tiles+j`，反解为 `i=task//n_tiles`、`j=task%n_tiles`：

```text
                j=0       j=1       j=2
               0:128    128:256    256:300
 i=0   0:128      0          1          2
 i=1 128:256      3          4          5
 i=2 256:384      6          7          8
 i=3 384:500      9         10         11
```

例如任务 11 写 `out[384:500,256:300]`，需要的输入为 `A[384:500,0:128]` 和 `B[0:128,256:300]`。它的有效输出是 `116×44`。所有输出矩形互不重叠，并且合起来恰好覆盖 500×300。

### 连续分配给各个核

取 `tiles_per_core = ceil(total/core_num)`，第 c 个核领取 `[start,end)`：

```text
start = min(c * tiles_per_core, total)
end   = min((c + 1) * tiles_per_core, total)
```

start 和 end 都截断到 total，空闲核得到 `[total,total)`。对本例分配如下：
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">启动核数</th><th style="text-align:left;">每核任务上限</th><th style="text-align:left;">各核区间</th></tr>
<tr><td>4</td><td>3</td><td>核 0～3：[0,3)、[3,6)、[6,9)、[9,12)</td></tr>
<tr><td>8</td><td>2</td><td>核 0～5：[0,2)、[2,4)、…、[10,12)；核 6～7：[12,12)</td></tr>
</table>

下图用 **20 个任务、3 个核** 展示同一分配规则；本节 12 个任务的具体区间以上表为准。

<img style="display:block; margin-left:0; margin-right:auto;" src="./images/pro_multicore_contiguous_grid.svg" alt="输出 Tile 编号和按核连续分配" width="700px">



连续分配容易推导，也使同一个核的任务具有局部性，但不保证所有核都忙。例如 12 个任务启动 8 核，最后两核空闲；100 个任务启动 32 核时，只有前 25 核各处理 4 个任务。增加启动核数不等于吞吐一定提高，尾块工作量也可能与整块不同。

本节是纯 Cube Kernel，直接使用 `pl.get_block_idx()` 获得 AIC 编号。

下面用普通整数计算展示任务分配，方便在写 Kernel 前检查边界；此单元可独立运行，只验证切分算术：


In [ ]:
m, n, cores = 500, 300, 8
m_tiles, n_tiles = (m + 127) // 128, (n + 127) // 128
total = m_tiles * n_tiles
per_core = (total + cores - 1) // cores
assigned = []
for core in range(cores):
    start = min(core * per_core, total)
    end = min((core + 1) * per_core, total)
    tasks = list(range(start, end))
    assigned.extend(tasks)
    print(f'core {core}: [{start}, {end}), tasks={tasks}')
assert assigned == list(range(total)), '任务遗漏或重复'


## 3. 尾块：5 个 Tile 的有效形状必须一致

任务 11 的有效 `M=116,N=44,K=128`，沿数据路径应设置：
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">Tile</th><th style="text-align:left;">物理 shape</th><th style="text-align:left;">当前 valid_shape</th></tr>
<tr><td>A 的 L1 / L0A</td><td>[128,128]</td><td>[116,128]</td></tr>
<tr><td>B 的 L1 / L0B</td><td>[128,128]</td><td>[128,44]</td></tr>
<tr><td>结果 L0C</td><td>[128,128]</td><td>[116,44]</td></tr>
</table>


从 A 的 L1 到 L0A，M/K 范围相同；从 B 的 L1 到 L0B，K/N 范围相同；L0C 的有效范围恰好对应二者输出的 M/N。只改 L0C 会留下输入搬运越界的风险，只改 L1 又会让计算使用不匹配的有效范围。

Kernel 声明物理 `shape=[128,128]`，用 `valid_shape=[-1,-1]` 表示运行时设置有效形状。每次取得槽位后，都在 load、move、matmul 前重新设置 5 个 Tile；即使上一任务是尾块、下一任务是完整块，也不能保留旧值。

L0A/L0B/L0C 的 `compact=1` 控制按有效尺寸紧凑排布，不能替代 `set_validshape`，也不缩小槽位已预留的最大容量。L1 输入组仍各占 64 KiB，L0A/L0B 单槽各占 32 KiB，L0C 单槽占 64 KiB。

本例的 K 固定为 128，不存在 K 尾块；将 K 改成任意值，还需要把 04.02 的累加循环、有效 K 和必要的 padding 一起设计，不能仅修改 Tensor 标注。


## 4. 完整实现：动态 M/N 与多核尾块

下面的 Kernel 把前面的任务公式与有效形状组合起来，支持正的 M/N、固定 K=128，输入输出均为连续 ND Tensor。Host 必须保证 A/B 的 K 一致且输出为 `[M,N]`；`pl.DYNAMIC` 表示维度可在运行时变化，并不自动检查三个 Tensor 之间的数学关系。

与单 Tile 示例相比，有四处主要变化：从 Tensor 读取 m/n；按核计算任务区间；把任务编号换算成 GM 的行列起点；每轮按边界设置有效形状。每个任务只做一次 `matmul`，所以 L0C 每次都会重新初始化。


In [ ]:
@pl.jit(auto_mutex=True)
def mm_dynamic(
    a: pl.Tensor[[pl.DYNAMIC, 128], pl.DT_FP16],
    b: pl.Tensor[[128, pl.DYNAMIC], pl.DT_FP16],
    out: pl.Tensor[[pl.DYNAMIC, pl.DYNAMIC], pl.DT_FP32],
):
    m = a.shape[0]
    n = b.shape[1]
    core_id = pl.get_block_idx()  # Pure Cube kernel: one task interval per AIC.
    core_num = pl.get_block_num()
    m_tiles = (m + 127) // 128
    n_tiles = (n + 127) // 128
    total = m_tiles * n_tiles
    tiles_per_core = (total + core_num - 1) // core_num
    start = pl.min(core_id * tiles_per_core, total)
    end = pl.min((core_id + 1) * tiles_per_core, total)

    a_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[-1, -1]),
        addrs=0x00000, mutex_ids=[0, 1])
    b_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[-1, -1]),
        addrs=0x10000, mutex_ids=[2, 3])
    a_left = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Left, valid_shape=[-1, -1],
                         compact=1),
        addrs=0x0000, mutex_ids=[4])
    b_right = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Right, valid_shape=[-1, -1],
                         compact=1),
        addrs=0x0000, mutex_ids=[5])
    acc = pl.make_tile_group(
        type=pl.TileType(shape=[128, 128], dtype=pl.DT_FP32,
                         target_memory=pl.MemorySpace.Acc, valid_shape=[-1, -1],
                         compact=1, fractal=1024),
        addrs=0x0000, mutex_ids=[6])

    with pl.section_cube():
        for task in pl.range(start, end, 1):
            i = task // n_tiles
            j = task % n_tiles
            row = i * 128
            col = j * 128
            valid_m = pl.min(128, m - row)
            valid_n = pl.min(128, n - col)
            cur_a = a_l1.next()
            cur_b = b_l1.next()
            al = a_left.current()
            br = b_right.current()
            ac = acc.current()
            pl.set_validshape(cur_a, [valid_m, 128])
            pl.set_validshape(cur_b, [128, valid_n])
            pl.set_validshape(al, [valid_m, 128])
            pl.set_validshape(br, [128, valid_n])
            pl.set_validshape(ac, [valid_m, valid_n])
            pl.load(cur_a, a, [row, 0])
            pl.load(cur_b, b, [0, col])
            pl.move(al, cur_a)
            pl.move(br, cur_b)
            pl.matmul(ac, al, br)
            pl.store(out, ac, [row, col])


### 沿任务 11 走一遍

当 4 核启动时，核 3 领取任务 9、10、11。进入任务 11 后：

1. `i=11//3=3`，`j=11%3=2`，所以 `row=384,col=256`。
2. `valid_m=min(128,500−384)=116`，`valid_n=min(128,300−256)=44`。
3. 设置 A 为 `[116,128]`、B 为 `[128,44]`、C 为 `[116,44]`。
4. 从 A 的 `[384,0]` 和 B 的 `[0,256]` 搬入，结果写回输出的 `[384,256]`。

这里 `load/store` 接受元素起点，因此必须乘 128。若使用 `load_tile/store_tile`，offset 的单位则是 Tile 块编号，不能混用两套坐标。

L1 使用 `next()` 轮转两个槽位，L0 使用 `current()` 复用单槽；自动同步管理同一槽位的读写依赖。地址预算与核数无关，因为这些是每个核自身的片上 Buffer，而 GM 的输出区域由任务编号负责隔离。


### 用多个形状检查同一份实现

只测 512×300，会漏掉 M 尾块；只测 500×300 的 4 核，也不会覆盖空闲核。下面依次检查任务少于核数、整除形状、只有 N 尾块，以及 M/N 双尾块。对 500×300 分别用 1、4、8 核运行，比较不同切分下的正确性。

这些调用要求目标设备支持相应启动核数；本节不测性能。每个用例输出 PASS 只表示该次精度校验通过。


In [ ]:
cases = [(128, 128, 4), (256, 256, 4), (512, 300, 4),
         (500, 300, 1), (500, 300, 4), (500, 300, 8)]
for m, n, cores in cases:
    torch.manual_seed(42)
    a = torch.randn(m, 128, device=device, dtype=torch.float16)
    b = torch.randn(128, n, device=device, dtype=torch.float16)
    out = torch.full((m, n), float('nan'), device=device, dtype=torch.float32)
    mm_dynamic[None, cores](a, b, out)
    torch.npu.synchronize()
    assert torch.isfinite(out).all().item(), f'M={m}, N={n}: 存在未写回位置或 NaN/Inf'
    golden = a.float() @ b.float()
    torch.testing.assert_close(out, golden, rtol=1e-2, atol=1e-2)
    # 单独观察最后一个输出 Tile，便于定位尾块问题。
    row = ((m - 1) // 128) * 128
    col = ((n - 1) // 128) * 128
    tail_error = (out[row:, col:] - golden[row:, col:]).abs().max().item()
    print(f'PASS M={m}, N={n}, cores={cores}, tail_max_abs={tail_error:.6g}')


## 5. 动态参数：DYNAMIC、TilingData 与 TilingKey 怎样分工

上面的 m/n 随输入变化，但 Tile 物理容量始终是 128×128。这正好说明“输入尺寸动态”和“片上分配尺寸动态”是两回事。
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">需求</th><th style="text-align:left;">适合的机制</th><th style="text-align:left;">本例中的含义</th></tr>
<tr><td>输入 M/N 随调用变化</td><td>Tensor 的 pl.DYNAMIC 维度</td><td>从 a.shape[0]、b.shape[1] 读取实际尺寸</td></tr>
<tr><td>Host 已算好循环边界、stride 等数值</td><td>TilingData：运行时 dataclass 参数</td><td>可传 m/n/任务数；每个 Kernel 最多一个，置于形参末尾</td></tr>
<tr><td>在有限几种 Tile 尺寸或调度方案间选择</td><td>TilingKey：编译期配置</td><td>例如 Tile_M 选择 64 或 128，各生成对应编译实例</td></tr>
</table>


以下仅展示参数定义，帮助区分两类值；当前 `mm_dynamic` 已从 Tensor shape 获取所需信息，不需要额外传入它们：

```python
from dataclasses import dataclass
from pypto_pro.runtime.tilingkey import TilingKeyField

@dataclass
class MatmulTiling:
    m: int
    n: int
    total_tiles: int

class MatmulKey:
    TileM = TilingKeyField(bits=1, values=[64, 128])

tiling = MatmulTiling(m=500, n=300, total_tiles=12)
key = {"TileM": 128}
```

如果改为 `Ptr + TilingData` 路线，可在 Kernel 中按 `tiling.m/n` 用 `make_tensor` 构造视图，具体传参方式见第 2 章。要使用上面的 Key，Kernel 必须通过 `@pl.jit(tiling_key=MatmulKey, ...)` 绑定它，再用 `kernel[None, core_num, key](...)` 选择实现。Host 传的是候选值 128，不是其编码下标 1。

注意两条边界：TilingData 的运行时字段不能直接作为 `TileType.shape` 的物理分配尺寸；更换 TileM 后，也必须同步推导任务数、槽位容量和地址。仅增加一个 Key 字段，并不会自动完成这套规划。


### 从失败形状缩小排查范围

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">哪个用例失败</th><th style="text-align:left;">优先检查</th></tr>
<tr><td>128×128、4 核</td><td>空闲核是否得到空区间，是否有多个核写同一 Tile</td></tr>
<tr><td>256×256 通过，512×300 失败</td><td>N 方向向上取整、B/结果的有效列数、store 列坐标</td></tr>
<tr><td>512×300 通过，500×300 失败</td><td>A/结果的有效行数，L0A 是否同步设置 valid_shape</td></tr>
<tr><td>同一形状 1 核通过，多核失败</td><td>核编号、start/end、任务到输出区域的映射</td></tr>
</table>

先看出错区域和失败形状，再检查相应索引；不要首先放宽 `assert_close` 的容差。


## 6. 课后练习

1. （填空）100 个任务、32 核按本节公式连续分配，每核任务上限为 ______，实际有任务的核数为 ______。
2. （选择）纯 Cube Kernel 的核编号应直接使用：A. `get_block_idx()`；B. 固定除以 2；C. `next()`。
3. （推导）写出 500×300、Tile=128、8 核时的全部 start/end；对任务 11 写出 A、B、C 的 GM 区域及 5 个 Tile 的有效形状。
4. （实践）如果只把物理 Tile_M 从 128 改成 64，哪些公式、地址容量和校验项需要重新检查？M 从 500 改成 501 时，是否也需要改变物理 Tile_M？


In [ ]:
!cat ./answer/04.04_answer.txt


## 7. 本节小结

多核正确性首先依赖任务覆盖完整、输出不重叠；尾块正确性依赖搬入、计算、写回共享同一组有效范围。动态输入尺寸可以改变循环与有效形状，但不能任意改变已经声明的物理容量。

进入 [章节实践](./04.05_chapter_practice.ipynb)，独立完成单 Tile、K 累加和动态多核三个版本，并用给定校验器检查这些边界。
